# TradeSense · Component 03 · Stage D · Notebook D1
### Loader, leak guards, baselines B1 / B2 / B4 and the B3 MLP (validation only)

**What this notebook does.** Reads the frozen Stage C tensors (`TGNN_StageC_GraphFeatures_v2`), builds a loader that **refuses test days**, checks the loader for leakage, then scores the baselines on the **validation month (July)** and trains the B3 MLP (4 pre-listed configurations on one seed, then the best one on 3 seeds). Nothing here opens an August, September or October day.

**What it does NOT do.** No TGNN (that is D2). No test days (that is D3). It never writes into Stage C or any audited folder. It writes only a new folder `TGNN_StageD_vN` on Drive.

**Rules written before any run (Protocol v2, section 14).** Target = scaled `log_qty` of the 50 nodes and scaled `log_spread_ticks` at interval t+1 (51 values). Metric = mean absolute error, equal weight per target value. Training = random 10% of the training windows per epoch, at most 20 epochs, early stopping on validation MAE with patience 3, 4 configurations (hidden 64 or 128, learning rate 1e-3 or 3e-4) tuned with one seed, best configuration run with 3 seeds, seeds averaged per window before any statistic.

**One implementation choice, stated openly.** Every learned model (B3 now, M0 and M1 in D2) outputs *the value at t plus a learned change*, starting exactly at persistence. This is the same for all learned models, so the comparison stays fair; without it the learned models would waste training just re-learning "copy the last value".

**B4 interpretation.** B4 is the rolling z-score of spread and top-5 imbalance. As a predictor it uses the z-score centre (the window mean) for the spread target; it has no node-target prediction, so it is judged on the spread column only. Its anomaly score (max absolute z of the two) is saved for later use.

**If Colab disconnects or the GPU quota stops:** set `RESUME_DIR` in cell 1 to the folder name printed in cell 3 and press *Run all*. Finished work is skipped, an interrupted training run continues from its last epoch file, nothing is overwritten.

**Use a T4 GPU runtime** (Runtime → Change runtime type → T4). On a CPU runtime the resource test fails by design.

## 1 · Settings (the only cell you may need to edit)

In [ ]:
import os, gc, sys, json, time, shutil, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), f'Drive folder not found: {BASE}'
STAGEC_NAME = 'TGNN_StageC_GraphFeatures_v2'          # frozen Stage C output (read only)
RESUME_DIR = None            # e.g. 'TGNN_StageD_v1' to continue an unfinished D1 run; None = start a new versioned folder
# ---- budget and rules, fixed BEFORE any run (Protocol v2 section 14.3) ----
FRAC = 0.10; MAX_EPOCHS = 20; PATIENCE = 3; BATCH = 256
SEED_TUNE = 100; SEEDS = [101, 102, 103]
MAX_B3_EPOCH_MIN = 30        # resource rule: one B3 epoch (10% of training windows + full validation) must fit in this many minutes
MIN_FREE_GPU_GB = 3.0        # GPU memory that must stay free after the train + validation tensors are loaded
EXPECTED = {'train_windows': 3453034, 'val_windows': 1725989}      # numbers recorded in Protocol v2 section 14.1
print('Stage C folder:', STAGEC_NAME, '| budget:', dict(frac=FRAC, max_epochs=MAX_EPOCHS, patience=PATIENCE, batch=BATCH, seeds=SEEDS))

Mounted at /content/drive
Stage C folder: TGNN_StageC_GraphFeatures_v2 | budget: {'frac': 0.1, 'max_epochs': 20, 'patience': 3, 'batch': 256, 'seeds': [101, 102, 103]}


## 2 · The Stage D library (written to the Colab working folder, not to Drive)

Loader with the test-day guard, evaluation harness, baselines, B3 MLP, resumable trainer, code tests. It only saves `staged_lib.py`.

In [ ]:
%%writefile staged_lib.py
"""TradeSense C03 - Stage D library (D1: loader, leak guards, baselines B1-B4, B3 MLP, resumable trainer).
Reads ONLY the frozen Stage C tensors. Never writes to Stage C. Test days are refused until a freeze file exists."""
import os, json, time, math, hashlib, shutil
from pathlib import Path
import numpy as np
import torch, torch.nn as nn

SEQ = 50; NN = 50; NCH = 6; GCH = 3; NT = 51; LOGQ = 2; GLS = 0; GIMB = 1
SYMS = ['BTCUSDT', 'ETHUSDT']
SPLITS = {'train': ['2024-05-01', '2024-06-01'], 'val': ['2024-07-01'], 'test': ['2024-08-01', '2024-09-01', '2024-10-01']}
def keys_of(split): return [f'{s}_{d}' for d in SPLITS[split] for s in SYMS]

# ---------------------------------------------------------------- small helpers
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def _j(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, Path): return str(o)
    raise TypeError(type(o))
def canonical_sha(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, default=_j).encode()).hexdigest()
def dump_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=1, default=_j)); os.replace(tmp, path)
def put_json(obj, path):
    """write once; if a different file already exists it is NEVER replaced: the new content goes to <name>.rN.json"""
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) == json.loads(json.dumps(obj, default=_j)): return path
        n = 1
        while path.with_name(f'{path.stem}.r{n}{path.suffix}').exists(): n += 1
        path = path.with_name(f'{path.stem}.r{n}{path.suffix}')
    dump_json(obj, path); return path
def check_frozen(path, cfg):
    """first call writes the frozen config; later calls must match exactly, otherwise resuming is refused"""
    path = Path(path)
    if not path.exists(): dump_json({'config': cfg, 'sha256': canonical_sha(cfg)}, path); return 'written'
    old = json.loads(path.read_text())
    if old['sha256'] != canonical_sha(cfg): raise RuntimeError(f'frozen config differs from the saved one ({path}); do not change settings or code while resuming a run')
    return 'matches'

# ---------------------------------------------------------------- the loader (all tensors in one block, optionally on the GPU)
class Bank:
    def __init__(self, stagec_dir, keys, device='cpu', freeze_file=None, chunk=100_000):
        self.dir = Path(stagec_dir) / 'sets'; self.keys = list(keys); self.device = device; self.summ = {}; total = 0
        for k in self.keys:
            s = json.loads((self.dir / k / 'set_summary.json').read_text()); self.summ[k] = s
            if s['split'] == 'test' and not (freeze_file and Path(freeze_file).exists()):
                raise PermissionError(f'{k} is a TEST day: refused because the freeze file does not exist yet')
            total += int(s['n_intervals'])
        self.nodes = torch.empty((total, NN, NCH), dtype=torch.float16, device=device); self.glob = torch.empty((total, GCH), dtype=torch.float16, device=device)
        self.off = {}; wins = []; sid = []; o = 0
        for i, k in enumerate(self.keys):
            arr = np.load(self.dir / k / 'nodes.npy', mmap_mode='r'); g = np.load(self.dir / k / 'glob.npy', mmap_mode='r'); n = arr.shape[0]
            assert arr.shape[1:] == (NN, NCH) and g.shape == (n, GCH), f'{k}: unexpected tensor shape {arr.shape} {g.shape}'
            for a in range(0, n, chunk):
                b = min(a + chunk, n); self.nodes[o + a:o + b] = torch.from_numpy(np.array(arr[a:b])).to(device)
            self.glob[o:o + n] = torch.from_numpy(np.array(g)).to(device)
            w = np.load(self.dir / k / 'win_end.npy'); wins.append(w.astype(np.int64) + o); sid.append(np.full(len(w), i, np.int16)); self.off[k] = (o, n); o += n; del arr, g
        self.win = np.concatenate(wins); self.win_set = np.concatenate(sid); self.ar = torch.arange(-SEQ + 1, 1, device=device)
    def gb(self): return (self.nodes.numel() * 2 + self.glob.numel() * 2) / 1e9
    def batch(self, g):
        """g: LongTensor of GLOBAL end indices t. Input = intervals t-49..t (inclusive). Target = interval t+1 only."""
        idx = g[:, None] + self.ar
        xn = self.nodes[idx].float(); xg = self.glob[idx].float()
        yn = self.nodes[g + 1][:, :, LOGQ].float(); yg = self.glob[g + 1][:, GLS].float()
        y = torch.cat([yn, yg[:, None]], 1); last = torch.cat([xn[:, -1, :, LOGQ], xg[:, -1, GLS:GLS + 1]], 1)
        return xn, xg, y, last

# ---------------------------------------------------------------- evaluation harness (per-window losses are kept for the later bootstrap)
@torch.no_grad()
def evaluate(bank, predict, bs=4096, extra=None):
    n = len(bank.win); loss = np.zeros(n, np.float32); chg = np.zeros(n, np.float32); nch = np.zeros(n, np.int16); spr = np.zeros(n, np.float32); ex = []
    for a in range(0, n, bs):
        b = min(a + bs, n); g = torch.from_numpy(bank.win[a:b]).to(bank.device); xn, xg, y, last = bank.batch(g)
        err = (predict(xn, xg, last) - y).abs(); m = (y != last)
        loss[a:b] = err.mean(1).cpu().numpy(); chg[a:b] = (err * m).sum(1).cpu().numpy(); nch[a:b] = m.sum(1).cpu().numpy(); spr[a:b] = err[:, NN].cpu().numpy()
        if extra is not None: ex.append(extra(xn, xg, last).cpu().numpy())
    out = {'loss': loss, 'chg_sum': chg, 'n_chg': nch, 'spr': spr}
    if extra is not None: out['extra'] = np.concatenate(ex)
    return out
def summarize(bank, res):
    rows = {}
    for i, k in enumerate(bank.keys):
        m = bank.win_set == i
        rows[k] = {'n_windows': int(m.sum()), 'mae_all': float(res['loss'][m].mean()), 'mae_changed': float(res['chg_sum'][m].sum() / max(int(res['n_chg'][m].sum()), 1)),
                   'share_changed': float(res['n_chg'][m].sum() / (NT * max(int(m.sum()), 1))), 'mae_spread': float(res['spr'][m].mean())}
    return rows
def average_results(rs):
    """seeds are averaged FIRST (mean of the per-seed losses of each window), exactly as in the frozen rule"""
    return {'loss': np.mean([r['loss'] for r in rs], 0).astype(np.float32), 'chg_sum': np.mean([r['chg_sum'] for r in rs], 0).astype(np.float32),
            'n_chg': rs[0]['n_chg'], 'spr': np.mean([r['spr'] for r in rs], 0).astype(np.float32)}
def save_result(res, bank, path):
    path = Path(path)
    if path.exists(): raise FileExistsError(path)
    tmp = path.with_name(path.stem + '.tmp.npz'); np.savez_compressed(tmp, win=bank.win, win_set=bank.win_set, **{k: v for k, v in res.items() if k != 'extra'}); os.replace(tmp, path)

# ---------------------------------------------------------------- baselines
def pred_b1(xn, xg, last): return last
def ema_weights(span, device='cpu'):
    a = 2.0 / (span + 1.0); j = torch.arange(SEQ - 1, -1, -1, dtype=torch.float32, device=device)       # time index 0 = oldest, 49 = latest
    w = (1 - a) ** j; return w / w.sum()
SPANS = [1, 2, 3, 5, 8, 13, 21, 34, 50]
def make_b2(span_node, span_spread):
    def f(xn, xg, last):
        wn = ema_weights(span_node, xn.device); ws = ema_weights(span_spread, xn.device)
        pn = torch.einsum('t,btn->bn', wn, xn[:, :, :, LOGQ]); ps = torch.einsum('t,bt->b', ws, xg[:, :, GLS])
        return torch.cat([pn, ps[:, None]], 1)
    return f
@torch.no_grad()
def tune_b2(bank, bs=4096):
    """one span for node targets, one span for the spread target, chosen on validation only"""
    sn = np.zeros(len(SPANS)); ss = np.zeros(len(SPANS)); n = len(bank.win)
    for a in range(0, n, bs):
        b = min(a + bs, n); g = torch.from_numpy(bank.win[a:b]).to(bank.device); xn, xg, y, last = bank.batch(g)
        for i, sp in enumerate(SPANS):
            p = make_b2(sp, sp)(xn, xg, last); e = (p - y).abs(); sn[i] += float(e[:, :NN].sum()); ss[i] += float(e[:, NN].sum())
    sn /= n * NN; ss /= n; return {'span_node': SPANS[int(sn.argmin())], 'span_spread': SPANS[int(ss.argmin())], 'val_mae_node_by_span': dict(zip(SPANS, sn.tolist())), 'val_mae_spread_by_span': dict(zip(SPANS, ss.tolist()))}
def pred_b4(xn, xg, last):
    """B4: the z-score centre as a naive predictor. Spread target -> window mean (z = 0). Node targets are not part of B4 (persistence is used only so that a tensor exists; B4 is judged on the spread column only)."""
    return torch.cat([last[:, :NN], xg[:, :, GLS].mean(1, keepdim=True)], 1)
def b4_score(xn, xg, last):
    """rolling z-score anomaly score of spread and top-5 imbalance (max of the two absolute z), saved for later fusion/anomaly use"""
    def z(v): return ((v[:, -1] - v.mean(1)) / (v.std(1) + 1e-3)).abs()
    return torch.maximum(z(xg[:, :, GLS]), z(xg[:, :, GIMB]))

# ---------------------------------------------------------------- B3 MLP (the same output rule is used for every learned model: prediction = value at t + learned change)
class MLP(nn.Module):
    def __init__(s, h):
        super().__init__(); s.net = nn.Sequential(nn.Linear(SEQ * NN * NCH + SEQ * GCH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, NT))
        nn.init.zeros_(s.net[-1].weight); nn.init.zeros_(s.net[-1].bias)           # starts exactly at persistence
    def forward(s, xn, xg, last): return last + s.net(torch.cat([xn.flatten(1), xg.flatten(1)], 1))
def make_mlp(cfg): return MLP(cfg['hidden'])
GRID = [{'hidden': h, 'lr': lr} for h in (64, 128) for lr in (1e-3, 3e-4)]      # the 4 pre-listed configurations

# ---------------------------------------------------------------- resumable trainer (also used by D2 for the TGNN)
def _load_latest(out, model, opt):
    for p in sorted(out.glob('resume_e*.pt'), reverse=True):
        try: ck = torch.load(p, map_location='cpu', weights_only=False)
        except Exception: continue
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); torch.set_rng_state(ck['rng_cpu'])
        if ck.get('rng_cuda') is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(ck['rng_cuda'])
        return ck['state'], p.name
    return None, None
def fit(make_model, cfg, bank_tr, bank_va, out_dir, seed, frac=0.10, max_epochs=20, patience=3, bs=256, device='cpu', log=print, _halt_after=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if (out / 'done.json').exists(): return json.loads((out / 'done.json').read_text())
    assert all(bank_tr.summ[k]['split'] == 'train' for k in bank_tr.keys), 'training bank contains a non-training set'
    torch.manual_seed(seed); model = make_model(cfg).to(device); opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    state = {'epoch': 0, 'best': float('inf'), 'best_epoch': 0, 'bad': 0, 'hist': []}
    st, name = _load_latest(out, model, opt)
    if st is not None: state = st; log(f'  resumed from {name} (epoch {state["epoch"]})')
    n = len(bank_tr.win)
    while state['epoch'] < max_epochs and state['bad'] < patience:
        e = state['epoch'] + 1; t0 = time.time(); rng = np.random.default_rng([seed, e]); pick = rng.permutation(n)[:int(n * frac)]
        model.train(); tot = torch.zeros((), device=device); cnt = 0
        for a in range(0, len(pick), bs):
            g = torch.from_numpy(bank_tr.win[pick[a:a + bs]]).to(device); xn, xg, y, last = bank_tr.batch(g)
            loss = (model(xn, xg, last) - y).abs().mean(); opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.detach() * len(g); cnt += len(g)
        t_train = time.time() - t0; model.eval(); res = evaluate(bank_va, lambda xn, xg, last: model(xn, xg, last)); val = float(res['loss'].mean())
        torch.save(model.state_dict(), out / f'weights_e{e:03d}.pt')
        state['epoch'] = e; state['hist'].append({'epoch': e, 'train_mae': float(tot / max(cnt, 1)), 'val_mae': val, 'train_seconds': round(t_train, 1), 'epoch_seconds': round(time.time() - t0, 1)})
        if val < state['best'] - 1e-9: state['best'] = val; state['best_epoch'] = e; state['bad'] = 0
        else: state['bad'] += 1
        tmp = out / f'resume_e{e:03d}.pt.tmp'
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'rng_cpu': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None, 'state': state}, tmp); os.replace(tmp, out / f'resume_e{e:03d}.pt')
        for p in out.glob('resume_e*.pt'):                                  # keep the two newest optimiser states; weights of every epoch stay
            if int(p.stem[8:]) < e - 1: p.unlink()
        with open(out / 'log.jsonl', 'a') as f: f.write(json.dumps(state['hist'][-1]) + '\n')
        log(f'  epoch {e}: train {state["hist"][-1]["train_mae"]:.5f} | val {val:.5f} | best {state["best"]:.5f} (e{state["best_epoch"]}) | {state["hist"][-1]["epoch_seconds"]} s')
        if _halt_after is not None and e >= _halt_after: return None
    done = {'cfg': cfg, 'seed': seed, 'best_val': state['best'], 'best_epoch': state['best_epoch'], 'epochs_run': state['epoch'], 'history': state['hist'], 'n_params': int(sum(p.numel() for p in model.parameters())),
            'mean_epoch_seconds': float(np.mean([h['epoch_seconds'] for h in state['hist']])), 'frac': frac, 'max_epochs': max_epochs, 'patience': patience, 'batch_size': bs}
    dump_json(done, out / 'done.json'); return done
def load_best(make_model, cfg, out_dir, device='cpu'):
    d = json.loads((Path(out_dir) / 'done.json').read_text()); m = make_model(cfg).to(device); m.load_state_dict(torch.load(Path(out_dir) / f'weights_e{d["best_epoch"]:03d}.pt', map_location=device)); m.eval(); return m, d

# ---------------------------------------------------------------- synthetic Stage C folder (for the code tests only)
def make_mock_stagec(root, n_int=2600, seed=0, bad_gap=(1200, 1290)):
    root = Path(root); sets = root / 'sets'; (root / 'config').mkdir(parents=True, exist_ok=True); rng = np.random.default_rng(seed)
    dump_json({'scaler': {'mock': True}, 'sha256': canonical_sha({'mock': True})}, root / 'config' / 'scaler_train_only.json')
    for split in ('train', 'val', 'test'):
        for day in SPLITS[split]:
            for sym in SYMS:
                k = f'{sym}_{day}'; d = sets / k; d.mkdir(parents=True, exist_ok=True)
                lq = np.zeros((n_int, NN), np.float32); cur = rng.normal(size=NN)
                for i in range(n_int):
                    ch = rng.random(NN) < 0.25; cur = np.where(ch, 0.9 * cur + rng.normal(scale=0.5, size=NN), cur); lq[i] = cur
                nodes = rng.normal(size=(n_int, NN, NCH)).astype(np.float32); nodes[:, :, 0] = np.where(np.arange(NN) < 25, -1, 1); nodes[:, :, LOGQ] = lq
                gl = rng.normal(size=(n_int, GCH)).astype(np.float32); gl[1:, GLS] = np.where(rng.random(n_int - 1) < 0.2, rng.normal(size=n_int - 1), 0); gl[:, GLS] = np.maximum.accumulate(np.zeros(n_int)) + np.cumsum(np.where(rng.random(n_int) < .1, rng.normal(scale=.3, size=n_int), 0))
                flags = np.zeros(n_int, np.uint8); flags[:SEQ + 20] = 16; flags[bad_gap[0]:bad_gap[1]] = 8
                c = np.concatenate([[0], np.cumsum((flags != 0).astype(np.int64))]); t = np.arange(SEQ, n_int - 1); win = t[(c[t + 2] - c[t - SEQ]) == 0].astype(np.int32)
                np.save(d / 'nodes.npy', nodes.astype(np.float16)); np.save(d / 'glob.npy', gl.astype(np.float16)); np.save(d / 'flags.npy', flags); np.save(d / 'win_end.npy', win)
                dump_json({'sym': sym, 'day': day, 'split': split, 'n_intervals': n_int, 'n_windows': int(len(win)), 'scaler_sha256': canonical_sha({'mock': True}), 'hashes': {f: sha256_file(d / f) for f in ('nodes.npy', 'glob.npy', 'flags.npy', 'win_end.npy')}}, d / 'set_summary.json')
    return root

# ---------------------------------------------------------------- E-tests (code on synthetic data)
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    root = make_mock_stagec(tmp / 'sc', n_int=2600); fz = tmp / 'freeze.json'
    # E01 test days are refused before the freeze file exists
    try: Bank(root, keys_of('test')[:1], freeze_file=fz); refused = False
    except PermissionError: refused = True
    dump_json({'x': 1}, fz); ok_after = True
    try: Bank(root, keys_of('test')[:1], freeze_file=fz)
    except Exception: ok_after = False
    fz.unlink(); add('E01', 'test days refused until the freeze file exists', refused and ok_after, f'refused={refused} allowed_after={ok_after}')
    btr = Bank(root, keys_of('train'), device='cpu'); bva = Bank(root, keys_of('val'), device='cpu')
    # E02 exact window indexing against direct array access
    rng = np.random.default_rng(1); worst = 0.0; okidx = True
    for i in rng.choice(len(btr.win), 40, replace=False):
        gt = int(btr.win[i]); k = btr.keys[int(btr.win_set[i])]; o, n = btr.off[k]; t = gt - o
        raw = np.load(btr.dir / k / 'nodes.npy'); rg = np.load(btr.dir / k / 'glob.npy')
        xn, xg, y, last = btr.batch(torch.tensor([gt]))
        okidx &= np.array_equal(xn[0].numpy(), raw[t - 49:t + 1].astype(np.float32)) and np.array_equal(xg[0].numpy(), rg[t - 49:t + 1].astype(np.float32))
        okidx &= np.array_equal(y[0, :NN].numpy(), raw[t + 1, :, LOGQ].astype(np.float32)) and float(y[0, NN]) == float(rg[t + 1, GLS]) and np.array_equal(last[0, :NN].numpy(), raw[t, :, LOGQ].astype(np.float32))
    add('E02', 'input = intervals t-49..t, target = interval t+1 only (40 random windows, exact)', okidx)
    # E03 the future cannot leak: overwrite interval t+1 and t+2 and check the input does not change
    gt = int(btr.win[100]); a = btr.batch(torch.tensor([gt]))[0].clone(); y0 = btr.batch(torch.tensor([gt]))[2].clone()
    sv = btr.nodes[gt + 1:gt + 3].clone(); btr.nodes[gt + 1:gt + 3] = 77.0; b = btr.batch(torch.tensor([gt]))
    same_x = torch.equal(a, b[0]); y_changed = not torch.equal(y0, b[2]); btr.nodes[gt + 1:gt + 3] = sv; add('E03', 'changing t+1 and t+2 does not change the input, only the target', same_x and y_changed)
    # E04 no window touches a flagged interval, and no window crosses a day boundary
    okf = True; okb = True
    for i, k in enumerate(btr.keys):
        fl = np.load(btr.dir / k / 'flags.npy'); w = np.load(btr.dir / k / 'win_end.npy'); okf &= all((fl[t - SEQ:t + 2] == 0).all() for t in w[::7]); okb &= int(w.min()) >= SEQ and int(w.max()) + 1 < btr.off[k][1]
    add('E04', 'windows contain no flagged interval and stay inside their day', okf and okb)
    # E05 baselines
    bs_ = 512; res1 = evaluate(bva, pred_b1, bs=bs_); man = []
    for i in rng.choice(len(bva.win), 30, replace=False):
        xn, xg, y, last = bva.batch(torch.tensor([int(bva.win[i])])); man.append((i, float((last - y).abs().mean())))
    okb1 = all(abs(res1['loss'][i] - v) < 1e-6 for i, v in man)
    res2 = evaluate(bva, make_b2(1, 1)); w = ema_weights(8); okb2 = np.allclose(res2['loss'], res1['loss'], atol=1e-6) and abs(float(w.sum()) - 1) < 1e-6 and bool((w >= 0).all()) and bool(w[-1] == w.max())
    add('E05', 'B1 equals a hand computation; B2 with span 1 equals B1; EMA weights are valid and causal', okb1 and okb2)
    # E06 changed-node mask by hand
    xn = torch.zeros(1, SEQ, NN, NCH); xg = torch.zeros(1, SEQ, GCH); last = torch.zeros(1, NT); y = torch.zeros(1, NT); y[0, 3] = 1.0; y[0, 50] = 2.0; p = torch.zeros(1, NT); err = (p - y).abs(); m = (y != last)
    add('E06', 'changed-target mask and per-window sums', int(m.sum()) == 2 and float((err * m).sum()) == 3.0)
    # E07 MLP starts exactly at persistence and can learn
    torch.manual_seed(0); mlp = MLP(16); xn, xg, y, last = btr.batch(torch.tensor(btr.win[:8])); p0 = mlp(xn, xg, last); okz = torch.equal(p0, last)
    opt = torch.optim.Adam(mlp.parameters(), lr=1e-3); l0 = float((mlp(xn, xg, last) - y).abs().mean().detach())
    for _ in range(60): l = (mlp(xn, xg, last) - y).abs().mean(); opt.zero_grad(); l.backward(); opt.step()
    add('E07', 'B3 starts at persistence and its loss falls on a small fit', okz and float(l.detach()) < l0 * 0.9, f'{l0:.4f}->{float(l.detach()):.4f}')
    # E08 resume gives exactly the same result as an uninterrupted run
    cfg = {'hidden': 8, 'lr': 1e-3}; quiet = lambda *a, **k: None
    d1 = fit(make_mlp, cfg, btr, bva, tmp / 'r_full', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet)
    fit(make_mlp, cfg, btr, bva, tmp / 'r_part', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet, _halt_after=1)
    d2 = fit(make_mlp, cfg, btr, bva, tmp / 'r_part', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet)
    sa = torch.load(tmp / 'r_full' / 'weights_e003.pt'); sb = torch.load(tmp / 'r_part' / 'weights_e003.pt')
    add('E08', 'a stopped-and-resumed run equals an uninterrupted run (weights and validation history)', all(torch.equal(sa[k], sb[k]) for k in sa) and [h['val_mae'] for h in d1['history']] == [h['val_mae'] for h in d2['history']])
    # E09 seed averaging and per-set summary by hand
    r = [{'loss': np.array([1., 2.], np.float32), 'chg_sum': np.array([2., 4.], np.float32), 'n_chg': np.array([1, 2], np.int16), 'spr': np.array([0., 1.], np.float32)}, {'loss': np.array([3., 4.], np.float32), 'chg_sum': np.array([4., 6.], np.float32), 'n_chg': np.array([1, 2], np.int16), 'spr': np.array([2., 3.], np.float32)}]
    av = average_results(r); add('E09', 'seeds are averaged per window before any statistic', np.allclose(av['loss'], [2, 3]) and np.allclose(av['chg_sum'], [3, 5]) and np.allclose(av['spr'], [1, 2]))
    # E10 sampler: deterministic per (seed, epoch), training bank holds only training sets, validation bank only validation
    p1 = np.random.default_rng([3, 2]).permutation(1000)[:100]; p2 = np.random.default_rng([3, 2]).permutation(1000)[:100]
    add('E10', 'epoch sample is deterministic per (seed, epoch); training bank = training sets only; validation bank = validation sets only', np.array_equal(p1, p2) and all(btr.summ[k]['split'] == 'train' for k in btr.keys) and all(bva.summ[k]['split'] == 'val' for k in bva.keys))
    # E11 frozen config
    try: check_frozen(tmp / 'fz.json', {'a': 1}); check_frozen(tmp / 'fz.json', {'a': 1}); refused = False; check_frozen(tmp / 'fz.json', {'a': 2})
    except RuntimeError: refused = True
    add('E11', 'a changed frozen config is refused', refused)
    # E12 put_json never overwrites
    put_json({'a': 1}, tmp / 'p.json'); put_json({'a': 2}, tmp / 'p.json'); add('E12', 'put_json keeps the first file and writes a changed one as .r1', json.loads((tmp / 'p.json').read_text()) == {'a': 1} and (tmp / 'p.r1.json').exists())
    # E13 B4 and tune_b2 run and give sane numbers
    tb = tune_b2(bva); sc_ = evaluate(bva, pred_b4, extra=b4_score); add('E13', 'B2 tuning and B4 run; B4 score is finite and non-negative', tb['span_node'] in SPANS and np.isfinite(sc_['extra']).all() and (sc_['extra'] >= 0).all())
    return out

Writing staged_lib.py


## 3 · Safety checks: device, Stage C folder, scaler, versioned output folder, frozen configuration

In [ ]:
import torch
sys.path.insert(0, str(Path.cwd())); import staged_lib as S
LIB_SHA = S.sha256_file(Path('staged_lib.py'))
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; GPU = torch.cuda.get_device_name(0) if DEV == 'cuda' else None
SC = BASE / STAGEC_NAME; assert (SC / 'sets').is_dir(), f'Stage C folder not found: {SC}'
sc_file = json.loads((SC / 'config' / 'scaler_train_only.json').read_text()); SC_SHA = sc_file['sha256']
HW = {'device': DEV, 'gpu': GPU, 'torch': torch.__version__, 'python': platform.python_version(), 'drive_free_gb': round(shutil.disk_usage(str(BASE)).free / 1e9, 2)}
if DEV == 'cuda': f_, t_ = torch.cuda.mem_get_info(); HW['gpu_mem_total_gb'] = round(t_ / 1e9, 2); HW['gpu_mem_free_gb'] = round(f_ / 1e9, 2)
print(json.dumps(HW, indent=1))
# ---- output folder (never overwrites) ----
if RESUME_DIR:
    OUT = BASE / RESUME_DIR; assert OUT.is_dir() and OUT.name.startswith('TGNN_StageD_v'), f'RESUME_DIR not found: {OUT}'; RESUMED = True
else:
    k = 1
    while (BASE / f'TGNN_StageD_v{k}').exists(): k += 1
    OUT = BASE / f'TGNN_StageD_v{k}'; OUT.mkdir(); RESUMED = False
for sub in ('config', 'd1/baselines', 'd1/b3', 'reports'): (OUT / sub).mkdir(parents=True, exist_ok=True)
FROZEN = {'stagec': STAGEC_NAME, 'scaler_sha256': SC_SHA, 'frac': FRAC, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'batch': BATCH, 'seed_tune': SEED_TUNE, 'seeds': SEEDS, 'grid': S.GRID, 'b2_spans': S.SPANS, 'library_sha256': LIB_SHA, 'splits': S.SPLITS, 'target': 'scaled log_qty of 50 nodes + scaled log_spread_ticks at t+1'}
print('frozen config:', S.check_frozen(OUT / 'config' / 'frozen_config_d1.json', FROZEN), '| output folder:', OUT.name, '| resumed:', RESUMED, '| library', LIB_SHA[:12], '| scaler', SC_SHA[:12])
FREEZE_FILE = OUT / 'config' / 'freeze_d3.json'       # does not exist in D1/D2: the loader therefore refuses every test day
assert not FREEZE_FILE.exists()

{
 "device": "cuda",
 "gpu": "Tesla T4",
 "torch": "2.11.0+cu130",
 "python": "3.13.15",
 "drive_free_gb": 16.1,
 "gpu_mem_total_gb": 15.64,
 "gpu_mem_free_gb": 15.53
}
frozen config: written | output folder: TGNN_StageD_v1 | resumed: False | library 85e7c1f444a4 | scaler 2bfe9d498569


## 4 · E-tests: the code on synthetic data (exact window indexing, future cannot leak, test days refused, resume equals uninterrupted run, baselines by hand)

In [ ]:
import tempfile
t0 = time.time()
with tempfile.TemporaryDirectory() as tmp_: ER = S.unit_tests(tmp_)
display(pd.DataFrame(ER, columns=['id', 'test', 'pass', 'detail']))
assert all(r[2] for r in ER), 'An E-test failed: stop here and send me this table.'
print(f'E-tests: {sum(r[2] for r in ER)}/{len(ER)} passed in {time.time()-t0:.0f} s')

,id,test,pass,detail
0,E01,test days refused until the freeze file exists,True,refused=True allowed_after=True
1,E02,"input = intervals t-49..t, target = interval t...",True,
2,E03,changing t+1 and t+2 does not change the input...,True,
3,E04,windows contain no flagged interval and stay i...,True,
4,E05,B1 equals a hand computation; B2 with span 1 e...,True,
5,E06,changed-target mask and per-window sums,True,
6,E07,B3 starts at persistence and its loss falls on...,True,0.0820->0.0329
7,E08,a stopped-and-resumed run equals an uninterrup...,True,
8,E09,seeds are averaged per window before any stati...,True,
9,E10,"epoch sample is deterministic per (seed, epoch...",True,


E-tests: 13/13 passed in 14 s


## 5 · Load the training and validation tensors (test days stay closed)

In [ ]:
t0 = time.time()
BTR = S.Bank(SC, S.keys_of('train'), device=DEV, freeze_file=FREEZE_FILE); BVA = S.Bank(SC, S.keys_of('val'), device=DEV, freeze_file=FREEZE_FILE)
print(f'train bank: {len(BTR.keys)} sets, {len(BTR.win):,} windows, {BTR.gb():.2f} GB | validation bank: {len(BVA.keys)} sets, {len(BVA.win):,} windows, {BVA.gb():.2f} GB | {time.time()-t0:.0f} s')
# the loader must refuse a test day right now
try: S.Bank(SC, S.keys_of('test')[:1], device='cpu', freeze_file=FREEZE_FILE); TEST_REFUSED = False
except PermissionError as e: TEST_REFUSED = True; print('test day refused as designed:', e)
assert TEST_REFUSED
if DEV == 'cuda': f_, t_ = torch.cuda.mem_get_info(); GPU_FREE_AFTER = f_ / 1e9; print(f'GPU memory free after loading: {GPU_FREE_AFTER:.2f} GB')
else: GPU_FREE_AFTER = None

train bank: 4 sets, 3,453,034 windows, 2.09 GB | validation bank: 2 sets, 1,725,989 windows, 1.05 GB | 102 s
test day refused as designed: BTCUSDT_2024-08-01 is a TEST day: refused because the freeze file does not exist yet
GPU memory free after loading: 12.27 GB


## 6 · D-tests on the real Stage C data

In [ ]:
DT = []
def T(i, nm, ok, det=''): DT.append((i, nm, bool(ok), str(det)[:300]))
sets_ok = all((SC / 'sets' / k / 'set_summary.json').exists() for k in S.keys_of('train') + S.keys_of('val') + S.keys_of('test'))
T('D01', 'Stage C folder: all 12 sets finished, scaler file present', sets_ok and (SC / 'config' / 'scaler_train_only.json').exists())
T('D02', 'same scaler hash in every Stage C set and equal to the saved scaler', all(BTR.summ[k]['scaler_sha256'] == SC_SHA for k in BTR.keys) and all(BVA.summ[k]['scaler_sha256'] == SC_SHA for k in BVA.keys), SC_SHA[:12])
T('D03', 'window counts equal the numbers in the protocol (train 3,453,034 / validation 1,725,989)', len(BTR.win) == EXPECTED['train_windows'] and len(BVA.win) == EXPECTED['val_windows'], f'{len(BTR.win):,} / {len(BVA.win):,}')
bad = []
for k in BTR.keys + BVA.keys:
    for fn in ('nodes.npy', 'glob.npy', 'win_end.npy'):
        h = S.sha256_file(SC / 'sets' / k / fn)
        if h != (BTR.summ.get(k) or BVA.summ[k])['hashes'][fn]: bad.append(f'{k}/{fn}')
T('D04', 'SHA-256 of the train and validation tensor files equals the Stage C record (test files are not touched)', not bad, bad[:3])
rng = np.random.default_rng(7); sm = BTR.win[rng.choice(len(BTR.win), min(20000, len(BTR.win)), replace=False)]; xs = BTR.nodes[torch.from_numpy(sm).to(DEV)].float(); gs = BTR.glob[torch.from_numpy(sm).to(DEV)].float()
mu = [float(xs[:, :, c].mean()) for c in (1, 2, 3, 4, 5)]; sd = [float(xs[:, :, c].std()) for c in (1, 2, 3, 4, 5)]
T('D05', 'scaled node channels in training windows: mean about 0 and std about 1 (frozen scaler applied)', all(abs(m) < 0.5 for m in mu) and all(0.3 < s < 3 for s in sd), f'mean {np.round(mu, 3).tolist()} std {np.round(sd, 3).tolist()}')
T('D06', 'all sampled train/validation values are finite and inside the float16 range', bool(torch.isfinite(xs).all() and torch.isfinite(gs).all()) and float(xs.abs().max()) < 6e4, f'abs max {float(xs.abs().max()):.1f}')
T('D07', 'only training sets in the training bank, only validation sets in the validation bank, no test day loaded', all(BTR.summ[k]['split'] == 'train' for k in BTR.keys) and all(BVA.summ[k]['split'] == 'val' for k in BVA.keys) and not (set(BTR.keys) | set(BVA.keys)) & set(S.keys_of('test')))
del xs, gs

## 7 · Baselines B1, B2, B4 on the validation month

In [ ]:
bl = OUT / 'd1' / 'baselines'; summ_f = bl / 'baselines_summary.json'
if summ_f.exists():
    BSUM = json.loads(summ_f.read_text()); print('baselines already computed - loaded')
else:
    t0 = time.time(); R1 = S.evaluate(BVA, S.pred_b1); TB2 = S.tune_b2(BVA); print('B2 spans chosen on validation:', TB2['span_node'], TB2['span_spread'])
    R2 = S.evaluate(BVA, S.make_b2(TB2['span_node'], TB2['span_spread'])); R4 = S.evaluate(BVA, S.pred_b4, extra=S.b4_score)
    for nm, r in (('b1', R1), ('b2', R2), ('b4', R4)): S.save_result(r, BVA, bl / f'{nm}_val.npz')
    np.save(bl / 'b4_anomaly_score_val.npy', R4['extra'].astype(np.float32))
    q = {str(p): float(np.quantile(R4['extra'], p)) for p in (.5, .9, .99, .999)}
    BSUM = {'B1': S.summarize(BVA, R1), 'B2': S.summarize(BVA, R2), 'B4': S.summarize(BVA, R4), 'b2_tuning': TB2, 'b4_score_quantiles': q, 'seconds': round(time.time() - t0, 1)}
    S.dump_json(BSUM, summ_f); print(f'baselines done in {time.time()-t0:.0f} s')
rows = []
for k in BVA.keys: rows.append({'set': k, 'B1 all': BSUM['B1'][k]['mae_all'], 'B2 all': BSUM['B2'][k]['mae_all'], 'B1 changed': BSUM['B1'][k]['mae_changed'], 'B2 changed': BSUM['B2'][k]['mae_changed'], 'share changed': BSUM['B1'][k]['share_changed'], 'spread B1': BSUM['B1'][k]['mae_spread'], 'spread B2': BSUM['B2'][k]['mae_spread'], 'spread B4': BSUM['B4'][k]['mae_spread']})
display(pd.DataFrame(rows).round(5)); print('B2 spans (node, spread):', BSUM['b2_tuning']['span_node'], BSUM['b2_tuning']['span_spread'], '| B4 score quantiles:', {k: round(v, 2) for k, v in BSUM['b4_score_quantiles'].items()})

B2 spans chosen on validation: 1 1
baselines done in 23 s


,set,B1 all,B2 all,B1 changed,B2 changed,share changed,spread B1,spread B2,spread B4
0,BTCUSDT_2024-07-01,0.04560,0.04560,0.21728,0.21728,0.20989,0.04040,0.04040,0.05117
1,ETHUSDT_2024-07-01,0.25996,0.25996,1.15852,1.15852,0.22439,0.01306,0.01306,0.01699


B2 spans (node, spread): 1 1 | B4 score quantiles: {'0.5': 0.97, '0.9': 2.25, '0.99': 4.51, '0.999': 6.84}


## 8 · B3 MLP: tune the 4 pre-listed configurations (validation only, one seed)

In [ ]:
b3 = OUT / 'd1' / 'b3'; TUNING = []
for i, cfg in enumerate(S.GRID):
    print(f'config {i}: {cfg}'); d = S.fit(S.make_mlp, cfg, BTR, BVA, b3 / f'tune_{i}', seed=SEED_TUNE, frac=FRAC, max_epochs=MAX_EPOCHS, patience=PATIENCE, bs=BATCH, device=DEV)
    TUNING.append({'config': i, **cfg, 'best_val': d['best_val'], 'best_epoch': d['best_epoch'], 'epochs_run': d['epochs_run'], 'mean_epoch_seconds': d['mean_epoch_seconds'], 'n_params': d['n_params']})
TUNING_DF = pd.DataFrame(TUNING); display(TUNING_DF.round(6))
BEST = int(TUNING_DF['best_val'].idxmin()); BEST_CFG = S.GRID[BEST]; S.put_json({'best_config_index': BEST, 'best_config': BEST_CFG, 'table': TUNING}, b3 / 'tuning.json'); print('best configuration (validation only):', BEST, BEST_CFG)

config 0: {'hidden': 64, 'lr': 0.001}
  epoch 1: train 0.16182 | val 0.15285 | best 0.15285 (e1) | 8.3 s
  epoch 2: train 0.16191 | val 0.15285 | best 0.15285 (e2) | 7.6 s
  epoch 3: train 0.16296 | val 0.15282 | best 0.15282 (e3) | 7.6 s
  epoch 4: train 0.16103 | val 0.15295 | best 0.15282 (e3) | 7.7 s
  epoch 5: train 0.16197 | val 0.15284 | best 0.15282 (e3) | 7.6 s
  epoch 6: train 0.16150 | val 0.15284 | best 0.15282 (e3) | 8.2 s
config 1: {'hidden': 64, 'lr': 0.0003}
  epoch 1: train 0.16172 | val 0.15277 | best 0.15277 (e1) | 7.7 s
  epoch 2: train 0.16185 | val 0.15278 | best 0.15277 (e1) | 8.0 s
  epoch 3: train 0.16290 | val 0.15278 | best 0.15277 (e1) | 7.6 s
  epoch 4: train 0.16090 | val 0.15278 | best 0.15277 (e1) | 7.7 s
config 2: {'hidden': 128, 'lr': 0.001}
  epoch 1: train 0.16186 | val 0.15284 | best 0.15284 (e1) | 8.6 s
  epoch 2: train 0.16191 | val 0.15285 | best 0.15284 (e1) | 8.3 s
  epoch 3: train 0.16301 | val 0.15286 | best 0.15284 (e1) | 8.9 s
  epoch 4: tr

,config,hidden,lr,best_val,best_epoch,epochs_run,mean_epoch_seconds,n_params
0,0,64,0.0010,0.152818,3,6,7.833333,977139
1,1,64,0.0003,0.152772,1,4,7.750000,977139
2,2,128,0.0010,0.152842,1,4,8.525000,1962419
3,3,128,0.0003,0.152774,3,6,8.583333,1962419


best configuration (validation only): 1 {'hidden': 64, 'lr': 0.0003}


## 9 · B3 MLP: best configuration with 3 seeds, seeds averaged per window

In [ ]:
R3 = []
for sd in SEEDS:
    print(f'seed {sd}'); d = S.fit(S.make_mlp, BEST_CFG, BTR, BVA, b3 / f'final_seed{sd}', seed=sd, frac=FRAC, max_epochs=MAX_EPOCHS, patience=PATIENCE, bs=BATCH, device=DEV)
    m, _ = S.load_best(S.make_mlp, BEST_CFG, b3 / f'final_seed{sd}', device=DEV); R3.append(S.evaluate(BVA, lambda xn, xg, last: m(xn, xg, last)))
RAVG = S.average_results(R3); out_npz = b3 / 'b3_val_seed_averaged.npz'
if not out_npz.exists(): S.save_result(RAVG, BVA, out_npz)
B3SUM = S.summarize(BVA, RAVG); S.put_json(B3SUM, b3 / 'b3_summary.json')
SEEDVAL = {sd: json.loads((b3 / f'final_seed{sd}' / 'done.json').read_text())['best_val'] for sd in SEEDS}
rows = []
for k in BVA.keys:
    b1 = BSUM['B1'][k]; r3 = B3SUM[k]; rows.append({'set': k, 'B1 all': b1['mae_all'], 'B3 all': r3['mae_all'], 'B3 vs B1 %': 100 * (1 - r3['mae_all'] / b1['mae_all']), 'B1 changed': b1['mae_changed'], 'B3 changed': r3['mae_changed'], 'changed vs B1 %': 100 * (1 - r3['mae_changed'] / b1['mae_changed']), 'spread B1': b1['mae_spread'], 'spread B3': r3['mae_spread']})
display(pd.DataFrame(rows).round(5)); print('per-seed best validation MAE:', {k: round(v, 6) for k, v in SEEDVAL.items()})

seed 101
  epoch 1: train 0.16156 | val 0.15278 | best 0.15278 (e1) | 8.1 s
  epoch 2: train 0.16125 | val 0.15278 | best 0.15278 (e2) | 7.6 s
  epoch 3: train 0.16181 | val 0.15277 | best 0.15277 (e3) | 7.8 s
  epoch 4: train 0.16086 | val 0.15277 | best 0.15277 (e3) | 7.8 s
  epoch 5: train 0.16164 | val 0.15278 | best 0.15277 (e3) | 7.6 s
  epoch 6: train 0.16082 | val 0.15278 | best 0.15277 (e3) | 7.8 s
seed 102
  epoch 1: train 0.16159 | val 0.15278 | best 0.15278 (e1) | 7.6 s
  epoch 2: train 0.16184 | val 0.15278 | best 0.15278 (e1) | 7.4 s
  epoch 3: train 0.16106 | val 0.15277 | best 0.15277 (e3) | 7.8 s
  epoch 4: train 0.16133 | val 0.15277 | best 0.15277 (e4) | 7.5 s
  epoch 5: train 0.16128 | val 0.15278 | best 0.15277 (e4) | 8.1 s
  epoch 6: train 0.16151 | val 0.15278 | best 0.15277 (e4) | 7.6 s
  epoch 7: train 0.16182 | val 0.15281 | best 0.15277 (e4) | 8.0 s
seed 103
  epoch 1: train 0.16173 | val 0.15277 | best 0.15277 (e1) | 8.4 s
  epoch 2: train 0.16069 | val 0.15

,set,B1 all,B3 all,B3 vs B1 %,B1 changed,B3 changed,changed vs B1 %,spread B1,spread B3
0,BTCUSDT_2024-07-01,0.04560,0.04563,-0.05839,0.21728,0.21728,0.00001,0.04040,0.04044
1,ETHUSDT_2024-07-01,0.25996,0.25999,-0.01008,1.15852,1.15852,0.00000,0.01306,0.01310


per-seed best validation MAE: {101: 0.152773, 102: 0.152772, 103: 0.152774}


## 10 · Resource tests

In [ ]:
RT = []
def Rt(i, nm, ok, det=''): RT.append((i, nm, bool(ok), str(det)[:300]))
Rt('R01', 'GPU runtime is in use', DEV == 'cuda', GPU)
Rt('R02', f'GPU memory: at least {MIN_FREE_GPU_GB} GB still free after loading train + validation tensors', (GPU_FREE_AFTER is not None and GPU_FREE_AFTER >= MIN_FREE_GPU_GB), GPU_FREE_AFTER)
ep_min = float(np.mean([r['mean_epoch_seconds'] for r in TUNING])) / 60
Rt('R03', f'one B3 epoch (10% of training windows + full validation) takes at most {MAX_B3_EPOCH_MIN} minutes', ep_min <= MAX_B3_EPOCH_MIN, f'measured {ep_min:.2f} min on {GPU}')
display(pd.DataFrame(RT, columns=['id', 'test', 'pass', 'detail']))

,id,test,pass,detail
0,R01,GPU runtime is in use,True,Tesla T4
1,R02,GPU memory: at least 3.0 GB still free after l...,True,12.268142592
2,R03,one B3 epoch (10% of training windows + full v...,True,measured 0.14 min on Tesla T4


## 11 · Decision, saved report (never overwritten) and the SHARE BLOCK — copy everything between the two ==== lines

In [ ]:
display(pd.DataFrame(DT, columns=['id', 'test', 'pass', 'detail']))
ALL = ER + DT + RT; n_pass = sum(r[2] for r in ALL); D1_OK = all(r[2] for r in ALL)
verdict = 'D1 COMPLETE' if D1_OK else 'D1 NEEDS ATTENTION'
rep = {'verdict': verdict, 'tests': {'E': ER, 'D': DT, 'R': RT}, 'stagec': STAGEC_NAME, 'scaler_sha256': SC_SHA, 'library_sha256': LIB_SHA, 'device': HW, 'budget': FROZEN, 'baselines': BSUM, 'b3_tuning': TUNING, 'b3_best_config': BEST_CFG, 'b3_seed_best_val': SEEDVAL, 'b3_val': B3SUM,
       'test_days_opened': False, 'measured_b3_epoch_minutes': ep_min, 'notes': ['learned models output value at t + learned change, starting at persistence (same for B3, M0, M1)', 'B4 judged on the spread column only']}
pth = S.put_json(rep, OUT / 'reports' / 'd1_report.json'); print('report:', pth.relative_to(BASE))
def mean_over(sm, key, sel):
    w = np.array([sm[k]['n_windows'] for k in sel], float); return float(np.sum([sm[k][key] * w_ for k, w_ in zip(sel, w)]) / w.sum())
sel = list(BVA.keys); share = {}
for nm, sm in (('B1', BSUM['B1']), ('B2', BSUM['B2']), ('B3', B3SUM)): share[nm] = (mean_over(sm, 'mae_all', sel), mean_over(sm, 'mae_changed', sel), mean_over(sm, 'mae_spread', sel))
share['B4'] = (None, None, mean_over(BSUM['B4'], 'mae_spread', sel))
lines = ['==== D1 SHARE BLOCK (copy everything between the two ==== lines) ====', f'verdict: {verdict} | tests passed {n_pass}/{len(ALL)} (E {sum(r[2] for r in ER)}/{len(ER)}, D {sum(r[2] for r in DT)}/{len(DT)}, R {sum(r[2] for r in RT)}/{len(RT)})',
         f'output folder: {OUT.name} | stage C: {STAGEC_NAME} | scaler {SC_SHA[:12]} | library {LIB_SHA[:12]} | gpu {GPU} | resumed {RESUMED}', 'test days opened: NO (loader refused a test day: %s)' % TEST_REFUSED,
         f'windows: train {len(BTR.win):,} | validation {len(BVA.win):,}', f'B2 spans (node, spread): {BSUM["b2_tuning"]["span_node"]}, {BSUM["b2_tuning"]["span_spread"]}', f'B3 best configuration: {BEST_CFG} | per-seed best val MAE: ' + ', '.join(f'{k}: {v:.6f}' for k, v in SEEDVAL.items()),
         'validation MAE (July, both symbols pooled by window count): all-target / changed-target / spread-only']
for nm in ('B1', 'B2', 'B3', 'B4'): a, c, s_ = share[nm]; lines.append(f'  {nm}: ' + (f'{a:.5f} / {c:.5f} / {s_:.5f}' if a is not None else f'n/a / n/a / {s_:.5f}'))
lines.append(f'B3 vs B1 on validation (descriptive only, not a pass/fail): all-target {100*(1-share["B3"][0]/share["B1"][0]):+.2f}% | changed-target {100*(1-share["B3"][1]/share["B1"][1]):+.2f}% | spread {100*(1-share["B3"][2]/share["B1"][2]):+.2f}%')
lines.append(f'measured B3 epoch: {ep_min:.2f} min | tuning table: ' + '; '.join(f'cfg{r["config"]} h{r["hidden"]} lr{r["lr"]:g} -> {r["best_val"]:.6f} (e{r["best_epoch"]})' for r in TUNING))
lines.append('failed tests: ' + (', '.join(r[0] for r in ALL if not r[2]) or 'none')); lines.append('==== END ====')
print('\n'.join(lines)); (OUT / 'reports' / 'd1_share_block.txt').write_text('\n'.join(lines)) if not (OUT / 'reports' / 'd1_share_block.txt').exists() else None

,id,test,pass,detail
0,D01,"Stage C folder: all 12 sets finished, scaler f...",True,
1,D02,same scaler hash in every Stage C set and equa...,True,2bfe9d498569
2,D03,window counts equal the numbers in the protoco...,True,"3,453,034 / 1,725,989"
3,D04,SHA-256 of the train and validation tensor fil...,True,[]
4,D05,scaled node channels in training windows: mean...,True,"mean [-0.001, -0.001, -0.001, -0.001, -0.0] st..."
5,D06,all sampled train/validation values are finite...,True,abs max 24.5
6,D07,"only training sets in the training bank, only ...",True,


report: TGNN_StageD_v1/reports/d1_report.json
==== D1 SHARE BLOCK (copy everything between the two ==== lines) ====
verdict: D1 COMPLETE | tests passed 23/23 (E 13/13, D 7/7, R 3/3)
output folder: TGNN_StageD_v1 | stage C: TGNN_StageC_GraphFeatures_v2 | scaler 2bfe9d498569 | library 85e7c1f444a4 | gpu Tesla T4 | resumed False
test days opened: NO (loader refused a test day: True)
windows: train 3,453,034 | validation 1,725,989
B2 spans (node, spread): 1, 1
B3 best configuration: {'hidden': 64, 'lr': 0.0003} | per-seed best val MAE: 101: 0.152773, 102: 0.152772, 103: 0.152774
validation MAE (July, both symbols pooled by window count): all-target / changed-target / spread-only
  B1: 0.15275 / 0.68774 / 0.02674
  B2: 0.15275 / 0.68774 / 0.02674
  B3: 0.15277 / 0.68774 / 0.02677
  B4: n/a / n/a / 0.03409
B3 vs B1 on validation (descriptive only, not a pass/fail): all-target -0.02% | changed-target +0.00% | spread -0.13%
measured B3 epoch: 0.14 min | tuning table: cfg0 h64 lr0.001 -> 0.1528

1103